# RAG Evaluation with DeepEval — Automated Quality Metrics

[![Open in Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/cyruslayo/castalia/blob/main/rag/evaluation_deep_eval.ipynb)

**A practical guide to measuring RAG pipeline quality with automated, reproducible metrics.**

Building a RAG pipeline is only half the battle. The other half is knowing whether it actually works — and *keeps* working as you swap models, change chunking strategies, or grow your knowledge base. This notebook introduces DeepEval, an open-source evaluation framework that automates RAG quality measurement across four critical dimensions: faithfulness, answer relevancy, contextual precision, and contextual recall.

| Component | Implementation |
|---|---|
| **LLM** | Qwen/Qwen3-8B (4-bit NF4 quantization) |
| **Embeddings** | BAAI/bge-base-en-v1.5 (768-dim, sentence-transformers) |
| **Vector Store** | FAISS (faiss-cpu) with numpy arrays |
| **Eval Framework** | DeepEval (faithfulness, relevancy, precision, recall) |
| **Data** | Synthetic inline knowledge base (no external files) |

> **Runtime:** Google Colab with T4 GPU. Estimated setup time: ~3 minutes.

## 1.1 — Learning Objectives

By the end of this notebook, you will:

- Understand the **RAG evaluation taxonomy** — retrieval quality vs. generation quality
- Use **DeepEval's metric suite** to evaluate a complete RAG pipeline end-to-end
- **Interpret evaluation results** to diagnose and fix specific pipeline failures
- Create **test cases with golden answers** for systematic, reproducible evaluation
- Connect **evaluation metrics to concrete pipeline improvements** (better chunking, reranking, prompt tuning)

## 1.2 — Why Manual RAG Evaluation Doesn't Scale

When you first build a RAG pipeline, manual spot-checking feels sufficient. You ask a few questions, glance at the retrieved documents, and confirm the answers look reasonable. But this approach has a fundamental flaw: **it only tests the cases you think to ask about.** Systematic failure modes — the ones that actually damage user trust — hide in the long tail of queries you never tried.

Consider a knowledge base with 10,000 document chunks. A manual review of 20 queries covers 0.2% of the interaction space. You might miss that your retriever consistently fails on temporal queries ("What changed in Q3?"), or that your LLM hallucinates when retrieved contexts are contradictory. These are not edge cases — they are entire categories of failure that affect real users every day.

As the corpus grows, failure modes multiply. New documents may introduce ambiguity with existing chunks. Embedding drift can silently degrade retrieval quality. A prompt that worked perfectly for 500 documents might start producing hallucinations at 5,000 documents because the context window is now saturated with marginally relevant text.

The solution is **automated, reproducible evaluation** — a test suite for your RAG pipeline that runs every time you make a change. Think of it as unit tests for AI: you define inputs, expected behaviors, and quality thresholds, then let the framework measure whether your pipeline meets the bar.

## 1.3 — The Evaluation Loop

Effective RAG development follows a tight feedback loop:

```
Build Pipeline --> Measure Quality --> Diagnose Failures --> Improve --> Measure Again
     |                                                              |
     +--------------------------------------------------------------+
```

Each iteration through this loop should answer specific questions: *Did changing the chunk size improve retrieval recall? Did the new system prompt reduce hallucination? Did adding a reranker help contextual precision?* Without metrics, these questions remain unanswerable — you are flying blind. With metrics, every change becomes a measurable experiment.

## 2.1 — RAG Evaluation Taxonomy

RAG evaluation operates along **two independent axes**: retrieval quality and generation quality. A pipeline can excel at one while failing at the other, and the fix depends entirely on which axis is broken.

**Retrieval quality** asks: *Did we fetch the right documents?* If a user asks about vacation policy and the retriever returns documents about expense reports, no amount of prompt engineering will save the answer. Retrieval quality is measured by precision (are the retrieved documents relevant?) and recall (did we retrieve all the relevant documents?).

**Generation quality** asks: *Did the LLM use the context correctly?* Even with perfect retrieval, the LLM might hallucinate facts not present in the context, ignore key evidence, or produce an answer that technically uses the context but doesn't address the question. Generation quality is measured by faithfulness (does the answer stick to the evidence?) and relevancy (does the answer address the question?).

The interaction between these two axes produces four distinct failure modes:

## 2.2 — The Four Failure Modes

| | Good Retrieval | Bad Retrieval |
|---|---|---|
| **Good Generation** | Correct, grounded answer | Plausible but wrong — coherent answer from irrelevant context |
| **Bad Generation** | Ignores evidence — correct docs retrieved but model hallucinates | Completely lost — wrong docs, wrong answer |

The most dangerous quadrant is **Good Retrieval + Bad Generation** — the retriever did its job, the evidence is right there in the context, but the LLM ignored it and made something up. This is pure hallucination, and it's invisible unless you explicitly measure faithfulness.

The most deceptive quadrant is **Bad Retrieval + Good Generation** — the answer reads well and sounds confident, but it's built on the wrong foundation. The model found something plausible to say about irrelevant context. Only by checking retrieval precision and recall can you catch this.

> **Key insight:** You need metrics on *both* axes. Evaluating only generation quality hides retrieval problems. Evaluating only retrieval quality hides hallucination problems.

## 2.3 — DeepEval's Metric Suite

DeepEval provides four core metrics for RAG evaluation. Each targets a specific failure mode and uses an LLM judge to compute scores. Here is how each metric works at a high level:

**Faithfulness** measures whether the generated answer contains only claims that are supported by the retrieved context. The algorithm extracts individual claims from the answer, then checks each claim against the context documents. The score is the fraction of claims that can be verified. A faithfulness score of 0.8 means 80% of the claims in the answer are grounded in the provided context — the remaining 20% are hallucinated.

**Answer Relevancy** measures whether the answer actually addresses the question asked. It works by generating synthetic questions from the answer and comparing them to the original question using semantic similarity. If the answer is on-topic, the synthetic questions should closely match the original. Low relevancy indicates tangential or off-topic responses — the model understood the context but missed the point of the question.

## 2.4 — Retrieval Metrics in DeepEval

**Contextual Precision** measures whether the most relevant documents are ranked highest in the retrieval results. It uses the expected output (golden answer) as a reference and checks whether the contexts that contain information needed to produce the golden answer appear at the top of the retrieved list. Think of it as a retrieval ordering metric — even if you retrieved the right documents, having irrelevant ones ranked above them dilutes the signal and wastes context window space.

**Contextual Recall** measures whether the retrieved context covers all the information needed to answer the question correctly. It decomposes the expected output into individual claims, then checks what fraction of those claims can be attributed to the retrieved context. A recall of 0.6 means the retriever only fetched enough information to cover 60% of the golden answer — the remaining 40% of needed information was never retrieved.

| Metric | Axis | Catches | Score = 1.0 means |
|---|---|---|---|
| Faithfulness | Generation | Hallucination | Every claim in the answer is supported by context |
| Answer Relevancy | Generation | Off-topic answers | The answer directly addresses the question |
| Contextual Precision | Retrieval | Poor ranking | Most relevant docs are ranked first |
| Contextual Recall | Retrieval | Missing documents | All information needed is in the context |

In [ ]:
!pip install -q deepeval sentence-transformers faiss-cpu \
    transformers torch bitsandbytes accelerate

## 3.1 — Imports and Configuration

In [ ]:
import torch
import json
import numpy as np
import faiss
from sentence_transformers import SentenceTransformer
from transformers import AutoModelForCausalLM, AutoTokenizer, BitsAndBytesConfig

## 3.2 — Knowledge Base

We create a synthetic knowledge base about a fictional company called **Meridian Technologies**. These 15 document chunks simulate realistic RAG scenarios — overlapping topics, specific numbers, policy details, and temporal references. This controlled setup lets us write golden answers and know exactly what the retriever should find.

In [ ]:
documents = [
    "Meridian Technologies offers 20 days of paid time off (PTO) per year for full-time employees. "
    "PTO accrues at a rate of 1.67 days per month and can be carried over up to 5 days into the next year.",

    "The company's remote work policy allows employees to work from home up to 3 days per week. "
    "Fully remote arrangements require director-level approval and a minimum of 6 months tenure.",

    "Meridian's health insurance plan covers medical, dental, and vision. The company pays 80% of premiums "
    "for employees and 50% for dependents. Open enrollment occurs annually in November.",

    "The engineering department uses a two-week sprint cycle. Sprint planning occurs on Mondays, "
    "daily standups are at 9:30 AM, and retrospectives are held on alternate Fridays.",

    "Meridian's 401(k) retirement plan matches employee contributions up to 6% of salary. "
    "The matching is 100% on the first 3% and 50% on the next 3%. Vesting is immediate.",

    "The company's code review policy requires at least two approvals before merging to main. "
    "Reviews should be completed within 24 hours. Automated CI/CD checks must pass before review.",

    "Meridian was founded in 2018 by Dr. Sarah Chen and Marcus Rivera. The company started as an "
    "AI consulting firm and pivoted to enterprise SaaS in 2020. It now has over 500 employees.",

    "New employees go through a 90-day onboarding program. The first week covers company culture, "
    "tools setup, and compliance training. Weeks 2-4 involve shadowing a senior team member.",

    "The annual performance review cycle runs from January to March. Reviews include self-assessment, "
    "peer feedback (minimum 3 peers), and manager evaluation. Ratings use a 5-point scale.",

    "Meridian's parental leave policy provides 16 weeks of paid leave for primary caregivers "
    "and 8 weeks for secondary caregivers. Leave can be taken within 12 months of birth or adoption.",

    "The IT security policy requires all employees to use two-factor authentication (2FA) for "
    "company systems. Passwords must be at least 14 characters and rotated every 90 days.",

    "Meridian's expense reimbursement policy covers business travel, client meals, and professional "
    "development. Expenses under $500 require manager approval; above $500 requires VP approval.",

    "The company's tech stack includes Python and TypeScript for backend services, React for "
    "frontend applications, PostgreSQL for primary data storage, and Redis for caching.",

    "Meridian's promotion criteria include demonstrated technical growth, cross-team impact, "
    "mentorship contributions, and alignment with company values. Promotions are reviewed biannually.",

    "The company provides a $2,000 annual learning stipend for courses, conferences, and books. "
    "Employees can also dedicate 10% of work time (one half-day per week) to learning projects.",
]

print(f"Knowledge base: {len(documents)} documents")

## 3.3 — Building the Embedding Index

We encode all documents using BGE-base and store them in a FAISS index with inner-product similarity. Since BGE produces normalized embeddings, inner product is equivalent to cosine similarity.

In [ ]:
embed_model = SentenceTransformer("BAAI/bge-base-en-v1.5")

doc_embeddings = embed_model.encode(documents, normalize_embeddings=True)
doc_embeddings = np.array(doc_embeddings, dtype="float32")

dimension = doc_embeddings.shape[1]
index = faiss.IndexFlatIP(dimension)
index.add(doc_embeddings)

print(f"FAISS index built: {index.ntotal} vectors, {dimension} dimensions")

## 3.4 — Retrieval Function

The retriever encodes the query, searches the FAISS index for the top-k nearest neighbors, and returns the corresponding document chunks.

In [ ]:
def retrieve(query: str, top_k: int = 3) -> list[str]:
    """Retrieve top-k documents for a query."""
    query_embedding = embed_model.encode(
        [query], normalize_embeddings=True
    ).astype("float32")
    scores, indices = index.search(query_embedding, top_k)
    return [documents[i] for i in indices[0]]


# Quick sanity check
test_contexts = retrieve("How much PTO do employees get?")
for i, ctx in enumerate(test_contexts):
    print(f"[{i+1}] {ctx[:80]}...")

## 3.5 — Loading the LLM

We load Qwen3-8B with 4-bit NF4 quantization to fit within Colab's T4 GPU memory. This model serves as both the RAG generator and (conceptually) illustrates the pipeline before we hand evaluation off to DeepEval's LLM judge.

In [ ]:
bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_compute_dtype=torch.float16,
)

model = AutoModelForCausalLM.from_pretrained(
    "Qwen/Qwen3-8B",
    quantization_config=bnb_config,
    device_map="auto",
)
tokenizer = AutoTokenizer.from_pretrained("Qwen/Qwen3-8B")

print(f"Model loaded: {model.config._name_or_path}")
print(f"Device: {model.device}")

## 3.6 — RAG Generation Function

The generation function takes a query and retrieved contexts, builds a grounded prompt, and generates an answer. We use greedy decoding (`do_sample=False`) for reproducibility.

In [ ]:
def generate_rag_answer(query: str, contexts: list[str]) -> str:
    """Generate an answer using retrieved contexts."""
    context_block = "\n".join(f"- {c}" for c in contexts)
    prompt = (
        "You are a helpful assistant. Answer the question based ONLY on the "
        "provided context. If the context does not contain enough information, "
        "say so explicitly.\n\n"
        f"Context:\n{context_block}\n\n"
        f"Question: {query}\n\n"
        "Answer:"
    )
    inputs = tokenizer(prompt, return_tensors="pt").to(model.device)
    with torch.no_grad():
        outputs = model.generate(
            **inputs,
            max_new_tokens=256,
            do_sample=False,
        )
    generated = outputs[0][inputs["input_ids"].shape[1]:]
    return tokenizer.decode(generated, skip_special_tokens=True).strip()

## 3.7 — Testing the RAG Pipeline

Before running formal evaluation, let's sanity-check the pipeline with a few example queries. We inspect both the retrieved contexts and the generated answers.

In [ ]:
sample_queries = [
    "How many days of PTO do employees get per year?",
    "What is the remote work policy?",
    "How does the 401(k) matching work?",
]

for q in sample_queries:
    contexts = retrieve(q)
    answer = generate_rag_answer(q, contexts)
    print(f"Q: {q}")
    print(f"A: {answer}")
    print(f"   Retrieved {len(contexts)} contexts")
    print("-" * 70)

## 4.1 — Introduction to DeepEval Test Cases

DeepEval organizes evaluation around **test cases** — structured objects that bundle everything needed to assess a single RAG interaction. Each test case contains:

- **`input`**: The user query
- **`actual_output`**: The answer your RAG pipeline produced
- **`expected_output`**: The golden answer — what a correct response should look like
- **`retrieval_context`**: The documents your retriever returned

By packaging these four components together, DeepEval can compute metrics that assess both retrieval and generation quality. The expected output serves as ground truth for recall-oriented metrics, while the retrieval context enables faithfulness checking.

In [ ]:
from deepeval.test_case import LLMTestCase
from deepeval.metrics import (
    FaithfulnessMetric,
    AnswerRelevancyMetric,
    ContextualPrecisionMetric,
    ContextualRecallMetric,
)

## 4.2 — Defining Golden Test Queries

A good evaluation suite covers diverse question types: factual lookups, comparisons, policy interpretations, and questions that require synthesizing information from multiple chunks. Each golden answer should be a concise, correct answer that a human expert would write.

In [ ]:
test_queries = [
    "How many days of PTO do full-time employees receive per year?",
    "What is the 401(k) matching policy at Meridian?",
    "How many approvals are required for a code review?",
    "What is the parental leave policy for primary caregivers?",
    "How much is the annual learning stipend?",
    "When was Meridian Technologies founded and by whom?",
    "What are the password requirements under the IT security policy?",
]

golden_answers = [
    "Full-time employees receive 20 days of paid time off per year, "
    "accruing at 1.67 days per month with up to 5 days carryover.",

    "Meridian matches 100% of the first 3% of salary contributed "
    "and 50% of the next 3%, for a maximum match of 6% of salary. "
    "Vesting is immediate.",

    "At least two approvals are required before merging to main. "
    "Reviews should be completed within 24 hours.",

    "Primary caregivers receive 16 weeks of paid parental leave. "
    "Leave can be taken within 12 months of birth or adoption.",

    "Meridian provides a $2,000 annual learning stipend for courses, "
    "conferences, and books. Employees can also use 10% of work time "
    "for learning projects.",

    "Meridian Technologies was founded in 2018 by Dr. Sarah Chen and "
    "Marcus Rivera. It started as an AI consulting firm and pivoted "
    "to enterprise SaaS in 2020.",

    "Passwords must be at least 14 characters long and rotated every "
    "90 days. Two-factor authentication is required for all company "
    "systems.",
]

print(f"Defined {len(test_queries)} test queries with golden answers")

## 4.3 — Running the RAG Pipeline on All Test Queries

We run each test query through the full RAG pipeline — retrieve, then generate — and store the results for evaluation. This gives us the raw material to build DeepEval test cases.

In [ ]:
results = []
for query, golden in zip(test_queries, golden_answers):
    contexts = retrieve(query)
    answer = generate_rag_answer(query, contexts)
    results.append({
        "query": query,
        "contexts": contexts,
        "answer": answer,
        "golden": golden,
    })
    print(f"Q: {query[:60]}...")
    print(f"A: {answer[:80]}...")
    print()

print(f"Generated answers for {len(results)} test queries")

## 4.4 — Creating DeepEval Test Cases

We package each result into an `LLMTestCase`. This is where the pipeline outputs meet the golden references — DeepEval will use these pairs to compute all four metrics.

In [ ]:
test_cases = []
for r in results:
    tc = LLMTestCase(
        input=r["query"],
        actual_output=r["answer"],
        expected_output=r["golden"],
        retrieval_context=r["contexts"],
    )
    test_cases.append(tc)

print(f"Created {len(test_cases)} DeepEval test cases")

## 4.5 — Initializing DeepEval Metrics

Each metric is initialized with a **threshold** — the minimum acceptable score — and a **model** that serves as the LLM judge. DeepEval uses this judge model to decompose answers into claims, verify them against context, and score the result.

> **Note:** DeepEval defaults to using GPT-4o as the judge model. This requires an OpenAI API key set as the `OPENAI_API_KEY` environment variable. If you do not have an API key, you can still study the test case structure and metric design — the conceptual understanding transfers to any evaluation framework.

In [ ]:
faithfulness = FaithfulnessMetric(threshold=0.7, model="gpt-4o")
answer_relevancy = AnswerRelevancyMetric(threshold=0.7, model="gpt-4o")
contextual_precision = ContextualPrecisionMetric(
    threshold=0.7, model="gpt-4o"
)
contextual_recall = ContextualRecallMetric(
    threshold=0.7, model="gpt-4o"
)

metrics = [
    faithfulness, answer_relevancy,
    contextual_precision, contextual_recall,
]
print("Metrics initialized:")
for m in metrics:
    print(f"  - {m.__class__.__name__} (threshold={m.threshold})")

## 4.6 — Running the Evaluation

DeepEval's `evaluate` function runs all metrics against all test cases and produces a comprehensive report. This is the moment of truth — the automated test suite for your RAG pipeline.

In [ ]:
from deepeval import evaluate

eval_result = evaluate(
    test_cases=test_cases,
    metrics=[
        faithfulness, answer_relevancy,
        contextual_precision, contextual_recall,
    ],
)

## 4.7 — Interpreting Evaluation Results

The evaluation output shows per-test-case scores for each metric, plus an overall pass/fail based on the thresholds you set. Here is how to read and act on the results:

- **Scores range from 0.0 to 1.0**, where 1.0 is perfect.
- A test case **passes** a metric if the score meets or exceeds the threshold (0.7 in our setup).
- The overall evaluation **passes** only if all test cases pass all metrics.

Use this diagnostic table to connect low scores to specific fixes:

| Metric | Low Score Means | Fix By |
|---|---|---|
| Faithfulness | Model is hallucinating beyond the context | Tighten prompt constraints, lower temperature, add citation instructions |
| Answer Relevancy | Answers are off-topic or tangential | Refine system prompt, check query understanding, verify context relevance |
| Contextual Precision | Irrelevant docs ranked above relevant ones | Add a reranker (cross-encoder), improve embedding model |
| Contextual Recall | Retriever misses relevant documents | Increase top_k, try different chunking strategies, use hybrid search |

## 4.8 — Extracting Scores Programmatically

Beyond the summary report, you can measure each metric individually to get detailed scores, reasons, and failure explanations. This is essential for building dashboards or tracking metrics over time.

In [ ]:
for i, tc in enumerate(test_cases):
    print(f"Test Case {i+1}: {tc.input[:60]}...")
    for metric in [faithfulness, answer_relevancy,
                    contextual_precision, contextual_recall]:
        try:
            metric.measure(tc)
            score = metric.score
            status = "PASS" if score >= metric.threshold else "FAIL"
            print(f"  {metric.__class__.__name__:>25s}: "
                  f"{score:.3f} [{status}]")
        except Exception as e:
            print(f"  {metric.__class__.__name__:>25s}: "
                  f"Error - {e}")
    print()

## 5.1 — Driving Improvements with Metrics

Metrics are not just report cards — they are **diagnostic tools** that point you toward specific improvements. Here is the improvement playbook for each metric:

**Low Faithfulness (hallucination)**
- Add explicit constraints to the prompt: "Answer ONLY based on the provided context"
- Reduce generation temperature to 0 (greedy decoding)
- Add citation instructions: "Reference the specific document that supports each claim"
- Consider shorter context windows to reduce noise

**Low Answer Relevancy (off-topic responses)**
- Check if the retriever is returning relevant documents (this might be a retrieval problem in disguise)
- Refine the system prompt to emphasize answering the specific question asked
- Add query rephrasing to handle ambiguous questions

**Low Contextual Precision (poor ranking)**
- Add a cross-encoder reranker after initial retrieval (see `rag/reranking.ipynb`)
- Try a more domain-specific embedding model
- Experiment with hybrid search (BM25 + dense retrieval, see `rag/fusion_retrieval.ipynb`)

**Low Contextual Recall (missing documents)**
- Increase `top_k` to retrieve more documents
- Use smaller chunk sizes so relevant passages are not buried in large chunks
- Try query expansion or HyDE (`rag/HyDe_Hypothetical_Document_Embedding.ipynb`)
- Implement hierarchical retrieval (`rag/hierarchical_indices.ipynb`)

## 5.2 — The Continuous Evaluation Mindset

The most important takeaway is that evaluation is not a one-time activity — it is a continuous process that should run every time you make a change to your pipeline. Treat your test cases like a regression test suite:

1. **Baseline**: Run evaluation on your current pipeline and record all scores.
2. **Change**: Modify one component (chunking, embeddings, prompt, top_k, etc.).
3. **Measure**: Run the same evaluation suite and compare scores.
4. **Decide**: If scores improve, keep the change. If they regress, revert.

Over time, your test suite should grow. Add test cases whenever you discover a failure in production. Add adversarial cases that probe known weaknesses. The stronger your test suite, the more confidently you can make changes.

## 5.3 — Exercises

**Exercise 1: Adversarial Test Cases**
Create 3 test cases designed to expose hallucination. Use questions where the answer is NOT in the knowledge base (e.g., "What is Meridian's stock price?") and verify that the model says it cannot answer rather than making something up. Measure faithfulness on these adversarial cases.

**Exercise 2: Metric Sensitivity to top_k**
Vary the `top_k` parameter from 1 to 10 and record all four metric scores for each value. Plot the results. At what point does increasing top_k start to hurt contextual precision? Does it always help contextual recall?

**Exercise 3: Custom Thresholds**
Different use cases demand different quality bars. For a medical knowledge base, you might require faithfulness >= 0.95 but accept answer relevancy >= 0.6. Set custom thresholds for a high-stakes scenario and re-run the evaluation. How many test cases pass now?

**Exercise 4: Prompt Engineering Loop**
Modify the `generate_rag_answer` prompt to include explicit citation instructions (e.g., "Cite the specific policy document for each claim"). Re-run evaluation and compare faithfulness scores before and after the prompt change.

## 5.4 — Key Takeaways

1. **Manual spot-checking does not scale.** Automated evaluation catches systematic failures that human reviewers miss, especially as the knowledge base grows.

2. **Evaluate both retrieval and generation.** A pipeline can have perfect retrieval but hallucinate anyway, or generate beautiful prose from the wrong documents. You need metrics on both axes.

3. **Faithfulness is the most critical metric** for trust. If the model hallucinates even once in a high-stakes domain, users lose confidence in the entire system.

4. **Golden answers enable ground-truth evaluation.** Investing time in writing correct reference answers pays off by enabling recall metrics and regression testing.

5. **Metrics are diagnostic, not just grades.** Each low score points to a specific improvement: prompt tuning, reranking, chunk size changes, or retrieval strategy upgrades.

6. **Evaluation is continuous.** Run your test suite after every pipeline change. Treat it like CI/CD for your RAG system.

7. **Start small, grow your test suite over time.** Begin with 5-10 golden test cases and add new ones whenever you discover a failure mode in production.

## 5.5 — References

- [DeepEval Documentation](https://docs.confident-ai.com/) — Official docs for the DeepEval framework
- [RAGAS: Automated Evaluation of Retrieval Augmented Generation](https://arxiv.org/abs/2309.15217) — Foundational paper on RAG evaluation metrics
- [Faithfulness and Hallucination in LLMs](https://arxiv.org/abs/2311.01477) — Survey of hallucination detection methods
- `rag/simple_rag.ipynb` — Build a RAG pipeline from first principles
- `rag/reranking.ipynb` — Improve contextual precision with cross-encoder reranking
- `rag/fusion_retrieval.ipynb` — Hybrid search to improve contextual recall
- `rag/HyDe_Hypothetical_Document_Embedding.ipynb` — Query expansion for better retrieval